# 10 — Capabilities & guardrails: steer and fence it

**Both** — you steer the agent with instructions that travel with its tools, and you fence it in with
blocks, approvals and budgets.

When you finish you can:

- bundle tools with the instructions that explain them in one `Capability`, for every run or for one run;
- log every tool call and block a dangerous one with a `Hooks` guardrail;
- pause a run until a person approves a risky tool call, then resume it;
- cap how many requests a run may spend with `UsageLimits`.

> This notebook runs **keyless**. Every model here is a `TestModel` or a `FunctionModel` (from 08), so CI
> runs it with no API key.

In [ ]:
from learningdantic import setup

MODEL = setup()

## 1. A capability = tools + the instructions that explain them

In 09 you scoped *what the agent can touch* with a toolset and `.filtered(...)`. The instructions that say
*how* to use those tools usually live somewhere else, and the two drift apart. A `Capability` keeps them
together: `Capability(instructions=..., toolsets=[...])` goes into `Agent(capabilities=[...])`, and the agent
gets both.

To see exactly what the model receives, the cell uses a tiny `FunctionModel` as a spy: instead of answering,
it replies with the tool names and the instructions it was given.

In [ ]:
from pydantic_ai import Agent, FunctionToolset
from pydantic_ai.capabilities import Capability
from pydantic_ai.messages import ModelResponse, TextPart
from pydantic_ai.models.function import AgentInfo, FunctionModel

desk_tools = FunctionToolset()


@desk_tools.tool_plain
def read_ticket(ticket_id: int) -> str:
    """Read one support ticket."""
    return f"ticket {ticket_id}: the printer on floor 2 is jammed"


@desk_tools.tool_plain
def close_ticket(ticket_id: int) -> str:
    """Mark a ticket as solved."""
    return f"ticket {ticket_id} closed"


@desk_tools.tool_plain
def delete_ticket(ticket_id: int) -> str:
    """Delete a ticket for good."""
    return f"ticket {ticket_id} deleted"


def spy(messages, info: AgentInfo) -> ModelResponse:
    """A fake model that reports what it was given instead of answering."""
    tools = [t.name for t in info.function_tools]
    return ModelResponse(parts=[TextPart(f"tools={tools}\ninstructions={info.instructions}")])


helpdesk = Capability(
    instructions="You run a helpdesk. Always call read_ticket before you answer about a ticket.",
    toolsets=[desk_tools.filtered(lambda ctx, td: td.name != "delete_ticket")],
)
agent = Agent(FunctionModel(spy), capabilities=[helpdesk])

r = await agent.run("What is ticket 7 about?")
print(r.output)

> ✅ **Checkpoint** — `tools=['read_ticket', 'close_ticket']` and the helpdesk instructions. `delete_ticket`
> was filtered out. The agent itself has no `instructions=` and no `toolsets=`: both came from the one
> capability. Give `helpdesk` to a second agent and it gets the same tools and the same rules.

## 2. A capability for one run only

`agent.run(..., capabilities=[...])` adds a capability to **that run only**. It is merged with the agent's
own capabilities and is gone on the next run. Use it for "this customer is upset" or "this user is an admin"
without building a second agent.

In [ ]:
upset = Capability(instructions="The customer is upset. Apologise before anything else.")

first = await agent.run("Where is my laptop?", capabilities=[upset])
second = await agent.run("Where is my laptop?")
print("--- run with the upset capability ---")
print(first.output)
print("--- next run ---")
print(second.output)

> ✅ **Checkpoint** — the first run's instructions are the helpdesk line, a blank line, then the apology
> line. The next run has only the helpdesk line: the per-run capability left no trace. The tools are the
> same both times.

Pydantic AI also ships ready-made capabilities. Two of them do 09's `.filtered` job, written as a capability:

- `MCP(local=pantry, allowed_tools=["check_stock"])` (from `pydantic_ai.capabilities`) connects 09's pantry
  server and keeps only the listed tools: the same result as 09's
  `pantry_tools.filtered(lambda ctx, td: td.name != "throw_out")`.
- `PrepareTools(fn)` is `.filtered` for all of the agent's tools at once: `fn(ctx, tool_defs)` returns the
  list of tools to keep.

More in the [capabilities docs](https://pydantic.dev/docs/ai/core-concepts/capabilities).

## 3. Hooks: log every tool call, block the dangerous one

Hiding a tool is the strongest fence. Sometimes you want a tool visible but guarded, or you just want a log
of everything the agent did. A `Hooks` capability runs your functions at fixed points in a run.
`before_tool_execute` runs after the model asks for a tool and before the tool runs:

- `return args` lets the call go ahead;
- `raise SkipToolExecution("...")` skips the tool; your text becomes the tool result the model sees.

A hook gets four arguments, and Pydantic AI passes the last three by name, so keep these names: `ctx` (the
run context from 05, with `ctx.deps`), `call` (the tool call: `call.tool_name`, `call.args`), `tool_def` (the
tool's definition) and `args` (the checked arguments).

`@guard.on.before_tool_execute(tools=["delete_ticket"])` limits a hook to the tools you name. Hooks are plain
`def`s, like tools. `TestModel` calls every tool it is offered, so this time the agent gets the whole
toolset, delete included.

In [ ]:
from pydantic_ai import SkipToolExecution
from pydantic_ai.capabilities import Hooks
from pydantic_ai.models.test import TestModel

audit = []
guard = Hooks()


@guard.on.before_tool_execute
def log_every_call(ctx, call, tool_def, args):
    audit.append(f"{call.tool_name}({args})")
    return args  # let the call go ahead


@guard.on.before_tool_execute(tools=["delete_ticket"])
def block_delete(ctx, call, tool_def, args):
    raise SkipToolExecution("BLOCKED: the assistant may not delete tickets.")


agent = Agent(TestModel(), toolsets=[desk_tools], capabilities=[guard])
r = await agent.run("Tidy up ticket 7.")
print("audit log:", sorted(audit))  # sorted, because tools asked for in one step run side by side
print("output   :", r.output)

> ✅ **Checkpoint** — the audit log has three entries, `delete_ticket` among them: the hook saw every request.
> In the output, `close_ticket` and `read_ticket` show their real results, but `delete_ticket` shows
> `BLOCKED: the assistant may not delete tickets.` The tool's body never ran. (`TestModel` fills in `0` for
> every argument.)

## 4. Ask a person before a risky tool runs

Some calls are fine *if* a person says yes. Mark the tool `requires_approval=True` and add
`DeferredToolRequests` to the output types. When the model calls that tool, the run stops early, and
`result.output` is a `DeferredToolRequests` whose `.approvals` list holds the calls waiting for a decision.

In [ ]:
from pydantic_ai import DeferredToolRequests

approval_agent = Agent(TestModel(), output_type=[str, DeferredToolRequests])
# New tool names, so they don't clash with the toolset functions defined above.


@approval_agent.tool_plain
def look_up_ticket(ticket_id: int) -> str:
    """Read one support ticket."""
    return f"ticket {ticket_id}: the printer on floor 2 is jammed"


@approval_agent.tool_plain(requires_approval=True)
def remove_ticket(ticket_id: int) -> str:
    """Delete a ticket for good."""
    return f"ticket {ticket_id} deleted"


paused = await approval_agent.run("Delete ticket 7.")
print("output type:", type(paused.output).__name__)
print("waiting for:", [call.tool_name for call in paused.output.approvals])

> ✅ **Checkpoint** — `output type: DeferredToolRequests` and `waiting for: ['remove_ticket']`. Watch out:
> `look_up_ticket` needs no approval, so it **already ran** in this first run. Only the risky call waits.

Now you decide. `paused.output.build_results(approvals={...})` takes a dict that maps each waiting call's
`tool_call_id` to `True` (approve) or `ToolDenied("reason")` (deny; the model sees your reason as the tool
result). `{call.tool_call_id: ... for call in waiting}` builds that dict in one line. Then run again with
the old messages plus your decisions, and the agent carries on from where it stopped.

In [ ]:
from pydantic_ai import ToolDenied

waiting = paused.output.approvals

denied = paused.output.build_results(
    approvals={call.tool_call_id: ToolDenied("A manager must approve deletes.") for call in waiting}
)
r = await approval_agent.run(message_history=paused.all_messages(), deferred_tool_results=denied)
print("denied   ->", r.output)

approved = paused.output.build_results(approvals={call.tool_call_id: True for call in waiting})
r = await approval_agent.run(message_history=paused.all_messages(), deferred_tool_results=approved)
print("approved ->", r.output)

> ✅ **Checkpoint** — after the denial, `remove_ticket`'s result is `A manager must approve deletes.`; after
> the approval it is `ticket 0 deleted`. Both resumes start from the same paused run, so you can try both.
> A real model would read that result and write its reply around it.

## 5. Put a budget on the run

A confused model can loop: call a tool, read the result, call it again, forever. `UsageLimits` puts a
ceiling on a run. When the next step would cross it, the run stops with `UsageLimitExceeded`, an error you
catch with `try` / `except`.

- `request_limit=` caps model requests. **If you pass nothing, a default of 50 still applies.**
- `tool_calls_limit=` caps tool calls; there are token and cost limits too
  ([usage limits docs](https://pydantic.dev/docs/ai/core-concepts/agent/#usage-limits)).

The `FunctionModel` below is stuck: it asks for `read_ticket` every single time.

In [ ]:
from pydantic_ai import UsageLimitExceeded, UsageLimits
from pydantic_ai.messages import ToolCallPart


def stuck(messages, info: AgentInfo) -> ModelResponse:
    """A fake model that never stops calling the same tool."""
    return ModelResponse(parts=[ToolCallPart("read_ticket", {"ticket_id": 7})])


agent = Agent(FunctionModel(stuck), toolsets=[desk_tools])

for limits in [UsageLimits(request_limit=3), None]:  # None = you passed no limits
    try:
        await agent.run("Summarise ticket 7.", usage_limits=limits)
    except UsageLimitExceeded as e:
        print(limits, "->", e)

> ✅ **Checkpoint** — two lines. `UsageLimits(request_limit=3) -> The next request would exceed the
> request_limit of 3...` and `None -> The next request would exceed the request_limit of 50...`. Without a
> limit of your own, a looping agent still stops, but only after 50 paid requests on a real model.

## 🏋️ Exercise — a helpdesk policy

Build one helpdesk agent with `read_ticket` and `delete_ticket`, plus a `Desk` deps dataclass (deps are from
05). Give it `audit_log: list[str] = field(default_factory=list)` (from `dataclasses`), which hands every new
`Desk` its own empty list; a plain `= []` is an error in a dataclass. Use `TestModel` / `FunctionModel` so it runs keyless.

1. **Audit + approval.** A `Hooks` capability appends every tool call's name to `ctx.deps.audit_log`, and
   `delete_ticket` needs approval. Run "Delete ticket 2.", deny the delete, then resume. Assert that the run
   paused on `delete_ticket`, that your reason reaches the final output, and that `delete_ticket` is **not**
   in the audit log (a denied tool never runs, so the hook never sees it).
2. **Tool-call cap.** Run the same agent with a stuck `FunctionModel` (pass `model=` to `run`) and
   `UsageLimits(tool_calls_limit=3)`. Assert that `UsageLimitExceeded` is raised and that the audit log holds
   exactly 3 entries.

*Stretch:* approve a delete only when the ticket id is on an allow-list. `call.args_as_dict()["ticket_id"]`
gives you each waiting call's id.

Solution: [`../solutions/10_guarded_helpdesk.ipynb`](../solutions/10_guarded_helpdesk.ipynb)

## Next →

[11 — Multi-agent](11_multi_agent.ipynb): split one big agent into a few small ones, each with one job.